# Module 7 Worksheet — Vector DB + Relational DB: Permission Filtering
**Corrected in this version:** `InHouseEmbeddings(model=MODEL_JINA)` → `InHouseEmbeddings()` — the corrected class takes no `model=` kwarg.

In [ ]:
import sys, os
sys.path.append(os.path.abspath("../../wrapper_fix"))      # folder containing the corrected inhouse_wrappers.py
sys.path.append(os.path.abspath("../../inhouse_rag_capstone"))  # folder containing your real inhouse_llm.py

from inhouse_llm import MODEL_QWEN3_14B, MODEL_QWEN3_30B, MODEL_MISTRAL, MODEL_LLAMA, MODEL_DEVSTRAL, MODEL_QWEN2_5_VL_7B
from inhouse_wrappers import get_chat_model, InHouseEmbeddings, build_vision_messages, llm_for
from langchain_core.messages import SystemMessage, HumanMessage

embedder = InHouseEmbeddings()

def ask(system_prompt, user_prompt, model=MODEL_QWEN3_14B, max_tokens=500):
    """Correctly-routed replacement for calling multimodal_chat() directly."""
    llm = get_chat_model(model=model, max_tokens=max_tokens)
    return llm.invoke([SystemMessage(content=system_prompt), HumanMessage(content=user_prompt)]).content

def ask_vision(system_prompt, user_prompt, image_base64, model=MODEL_QWEN2_5_VL_7B, max_tokens=500):
    """Correctly-routed, correctly-formatted multimodal call."""
    llm = get_chat_model(model=model, max_tokens=max_tokens)
    return llm.invoke(build_vision_messages(system_prompt, user_prompt, image_base64)).content

print("Setup OK")

In [ ]:
# pip install chromadb psycopg2-binary --break-system-packages
import chromadb

## 1. Setup: Chroma collection with an owner field in metadata

In [ ]:
client = chromadb.Client()  # in-memory for this worksheet

collection = client.get_or_create_collection("perm_demo")
docs = [
    "MCP standardizes tool calling.",
    "RAG grounds answers in retrieved context.",
    "Qwen3-14B is the default chat model.",
    "Internal salary bands for team_a.",
    "Internal salary bands for team_b.",
]
owners = ["public", "public", "public", "team_a", "team_b"]
ids = [f"doc_{i}" for i in range(len(docs))]
embeddings = embedder.embed_documents(docs)
collection.add(ids=ids, embeddings=embeddings, documents=docs,
               metadatas=[{"owner": o} for o in owners])

## 2. Reproducing the teaser bug: filter AFTER retrieval

In [ ]:
current_user_owner = "team_a"
query = "internal information"
q_vec = embedder.embed_query(query)

# WRONG: top-k first, filter after
results = collection.query(query_embeddings=[q_vec], n_results=2)
print("Top-2 before filtering:", results["documents"][0], results["metadatas"][0])

filtered = [d for d, m in zip(results["documents"][0], results["metadatas"][0])
            if m["owner"] in (current_user_owner, "public")]
print("After filtering (may be empty even though permitted docs exist!):", filtered)

## 3. The fix: filter DURING retrieval via `where`

In [ ]:
results_fixed = collection.query(
    query_embeddings=[q_vec], n_results=2,
    where={"owner": {"$in": [current_user_owner, "public"]}},
)
print("Top-2 WITH permission filter applied during search:", results_fixed["documents"][0])

## Teaser exercise
Add a `pip install psycopg2-binary` Postgres version of the same scenario: store owner/permission data in Postgres instead of Chroma metadata, and compare doing the permission check as a SQL `WHERE` clause vs filtering a Python list after the fact — same bug, same fix, different system.